In [1]:
!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [2]:
from langchain_core.documents import Document

In [3]:
sample_doc = Document(  # defining a doc 
    page_content="Hello World!",
    metadata={"source":"https://www.google.com"} 
)

In [4]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World!')

In [5]:
type(sample_doc)

langchain_core.documents.base.Document

In [8]:
# Text data
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("data/Python.txt", encoding="utf-8")

In [9]:
document = loader.load()

document # loaded the pdf 

In [10]:
# # PDF data
# from langchain_community.document_loaders.pdf import PyPDFLoader # pyMuPDFLoader for complex pdfs

# pdf_loader = PyPDFLoader("data/research2.pdf")

# document = pdf_loader.load()
# document

# INGESTION MODULE

In [11]:
# Data => module
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

## Documents 

In [12]:
def load_all_pdfs():
    folder_path = "data/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs :", num_docs)
    print("total pages :", len(all_docs))

    return all_docs

In [38]:
all_pdf_documents  = load_all_pdfs()

total pdfs : 2
total pages : 32


In [39]:
type(all_pdf_documents[1])

langchain_core.documents.base.Document

# CHUNKS

In [40]:
!pip install langchain_text_splitters

In [44]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents , chunk_size = 500 , chunk_overlap = 50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [45]:
chunks = split_docs(all_pdf_documents)

In [46]:
len(chunks)

320

In [47]:
#chunks

 ## Embedding

In [48]:
from sentence_transformers import SentenceTransformer

In [49]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):

        self.model_name = model_name
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print(
            "embedding dimensions=",
            self.model.get_sentence_embedding_dimension(),
        )

    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape:", embeddings.shape)
        return embeddings

In [50]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions= 384


/var/folders/1c/29w4nj612ns_rgx1ljtbvlnh0000gn/T/ipykernel_2526/4268907676.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.model.get_sentence_embedding_dimension(),


# Vector Store

In [51]:
# vector store manager
# 1. initialize vector store manager
# 2. vs initialize -> path , collection
# 3. store all the documents in the Vector store

In [52]:
import chromadb
import uuid # help us create indexes for our value

In [53]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        
        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")


        # store => ids, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

        print("total documents added in vector store=", len(documents_content))
        print("docs in collection:", self.collection.count())

In [54]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 493


In [55]:
# data => documents => chunks => embeddings => store in vector store
# now we are going to create embeddings form cunks

texts = [doc.page_content for doc in chunks]

emebedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, emebedding)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

embeddings shape: (320, 384)
total documents added in vector store= 320
docs in collection: 813


# Retrieval  Pipeline

In [57]:
from sklearn.metrics.pairwise import cosine_similarity

In [73]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store


    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs=[]
        
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [74]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [75]:
rag_retriever.retrieve("What is encoder decoder")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 4 documents


[{'id': 'doc_dc92f73a-d653-459d-99a7-f19629a2ca4b',
  'document': 'positional encodings in both the encoder and decoder stacks. For the base model, we use a rate of\nPdrop = 0.1.\n7',
  'metadata': {'published': '2017',
   'lastpage': '6008',
   'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin',
   'source': 'data/pdfs/research1.pdf',
   'eventtype': 'Poster',
   'producer': 'PyPDF2',
   'book': 'Advances in Neural Information Processing Systems 30',
   'page': 6,
   'publisher': 'Curran Associates, Inc.',
   'total_pages': 11,
   'editors': 'I. Guyon and U.V. Luxburg and S. Bengio and H. Wallach and R. Fergus and S. Vishwanathan and R. Garnett',
   'doc_index': 293,
   'moddate': '2018-02-12T21:22:10-08:00',
   'type': 'Conference Proceedings',
   'creator': 'PyPDF',
   'creationdate': '',
   'created': '2017',
   'language': 'en-US',
   'title': 'Attention is All you Need',
   'firstpage': '5998',
   '

# Integrate RAG with LLM's

## Open AI

In [77]:
API_KEY_OPENAI = "gareeb"

In [78]:
!pip install langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 13.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langchain-openai]


In [84]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key = API_KEY_OPENAI,
    model = "gpt-5.4",
    temperature=0.1, # low for fact based ans and high for creative ans like poem
    max_tokens = 1024
)

In [85]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query} """

    response = llm.invoke(prompt) # expecting a string as prompt
    return response.content

In [86]:
answer = generate_output("what is encoder-decoder?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 3 documents


OpenAIAuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: gareeb. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

In [87]:
answer

NameError: name 'answer' is not defined

## Groq

In [121]:
API_Key_GROQ= "grok_api_key"

In [105]:
!pip install langchain-groq

In [110]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=API_Key_GROQ,
    model="qwen/qwen3.6-27b",
    temperature=0.1,
    max_tokens=1024
)

In [111]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query} """

    response = llm.invoke([prompt.format(context=context, query=query)]) # expecting a list as prompt
    return response.content

In [116]:
answer = generate_output("what is RAG?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 3 documents


In [117]:
# from groq import Groq

# client = Groq(api_key=API_Key_GROQ)

# models = client.models.list()

# for model in models.data:
#     print(model.id)

# 🌟🌟🌟🌟to see what models you can use with this API key

In [118]:
print(answer)


<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Context:** The provided text is repetitive and fragmented. It mentions: "and speculate on upcoming trends and innovations. Our contributions are as follows: • In this survey, we present a thorough and systematic review of the state-of-the-art RAG methods, delineating its evolution through paradigms including naive RAG, arXiv:2312.10997v5 [cs.CL] 27 Mar 2024" (repeated 3 times).
   - **Query:** "what is RAG?"
   - **Task:** Use the given context to generate an answer for the query.

2.  **Evaluate Context vs. Query:**
   - The context mentions "RAG methods" and "naive RAG" as part of a survey on state-of-the-art methods. It also mentions an arXiv ID (2312.10997v5) and a date.
   - However, the context *does not define* what RAG stands for or what it is. It only refers to it as a set of methods/paradigms being reviewed.
   - I need to answer based *only* on the provided context, but I should also acknowledge the limi

 ## Claude

In [119]:
# do it on your own

In [120]:
# !pip install langchain-anthropic